In [1]:
from datasets import load_dataset
from data_structures import Example
from hierarchical_optimizer import HierarchicalOptimizer

GSM8K_METRICS = [
    {"name": "numeric_exact_match",  "weight": 0.8, "stage": 1},
    {"name": "numeric_token_f1",     "weight": 0.2, "stage": 1},
]

/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/kateav

In [2]:
import re

def _extract_gold_answer(answer_text: str) -> str:
    """Extract the numerical answer after #### from GSM8K answer field."""
    match = re.search(r'####\s*(.+)', answer_text)
    if match:
        return match.group(1).strip().replace(',', '')
    numbers = re.findall(r'-?[\d,]+\.?\d*', answer_text)
    if numbers:
        return numbers[-1].replace(',', '')
    return answer_text.strip()


def gsm8k_to_examples(data):
    examples = []
    for item in data:
        question = item["question"]
        full_answer = item["answer"]

        final_number = _extract_gold_answer(full_answer)

        reasoning = full_answer.split("####")[0].strip() if "####" in full_answer else ""

        input_text = f"Question: {question}"
        examples.append(Example(
            input_text=input_text,
            expected_output=final_number,
            metadata={
                "full_solution": full_answer,
                "reasoning": reasoning,
                "all_answers": [final_number],
                "task_type": "numeric_qa"
            },
        ))
    return examples


def get_gsm8k_data(train_num: int, val_num: int, test_num: int):
    ds_train = load_dataset('openai/gsm8k', 'main', split='train')
    ds_test = load_dataset('openai/gsm8k', 'main', split='test')

    ds_train = ds_train.shuffle(seed=42)
    ds_test = ds_test.shuffle(seed=42)

    split = ds_train.train_test_split(test_size=0.1, seed=42)
    ds_train = split['train']
    ds_val = split['test']

    train_split = ds_train.select(range(min(train_num, len(ds_train))))
    val_split = ds_val.select(range(min(val_num, len(ds_val))))
    test_split = ds_test.select(range(min(test_num, len(ds_test))))

    train_examples = gsm8k_to_examples(train_split)
    validation_examples = gsm8k_to_examples(val_split)
    test_examples = gsm8k_to_examples(test_split)

    return train_examples, validation_examples, test_examples


def data_fabric(dataset: str = 'gsm8k', train_num: int = 50, val_num: int = 50, test_num: int = 50):
    gsm8k_initial_prompt = (
        "Solve the following math word problem step by step. "
        "Show your reasoning clearly, then provide the final numerical answer. "
        "Output ONLY the final number as the last line of your response, "
        "with no additional text, units, or symbols."
    )

    train_examples, validation_examples, test_examples = get_gsm8k_data(
        train_num, val_num, test_num
    )
    return train_examples, validation_examples, test_examples, gsm8k_initial_prompt

In [3]:
train_examples, validation_examples, test_examples, initial_prompt = data_fabric()

print("Dataset prepared:")
print(f"  Train: {len(train_examples)} examples")
print(f"  Validation: {len(validation_examples)} examples")
print(f"  Test: {len(test_examples)} examples")

Dataset prepared:
  Train: 50 examples
  Validation: 50 examples
  Test: 50 examples


In [4]:
print("Initial prompt:")
print("-" * 60)
print(initial_prompt)
print("-" * 60)

Initial prompt:
------------------------------------------------------------
Solve the following math word problem step by step. Show your reasoning clearly, then provide the final numerical answer. Output ONLY the final number as the last line of your response, with no additional text, units, or symbols.
------------------------------------------------------------


In [5]:
TASK_DESCRIPTION = (
    "Grade-school math word problem solving: given a math word problem in natural language, "
    "reason step by step and produce the correct final numerical answer. "
    "The answer must be a single number with no units or extra text."
)

optimizer = HierarchicalOptimizer(
    metrics_config=GSM8K_METRICS,
    task_description=TASK_DESCRIPTION,
)


In [6]:
best_node = optimizer.optimize(
    initial_prompt=initial_prompt,
    train_examples=train_examples,
    validation_examples=validation_examples,
    test_examples=test_examples,
    save_dir="./optimization_results_gsm8k",
)

Evaluating initial prompt...
[diag] initial node: prompt_id=4619780f0089 len=229 chars
[diag] Prompt text: Solve the following math word problem step by step. Show your reasoning clearly, then provide the final numerical answer. Output ONLY the final number as the last line of your response, with no additional text, units, or symbols.
[diag] evaluate_node: node_id=bca65a3b-ab1a-4f73-bac1-7600bd9fb00c gen=0 source=initial split=validation stage=1 examples=50 seed_offset=0
[diag] evaluate_prompt: execute=True sample=True stage=1 incoming=50 will_use=50 weights={numeric_exact_match:0.8, numeric_token_f1:0.2} llm_calls_at_start=0
[diag] execute_prompt_batch: total=50 batch_size=10 n_batches=5 llm_calls_before=0
[diag]   batch 1/5: OK (10/50 done, llm_calls=0)
[diag]   batch 2/5: OK (20/50 done, llm_calls=0)
[diag]   batch 3/5: OK (30/50 done, llm_calls=0)
[diag]   batch 4/5: OK (40/50 done, llm_calls=0)
[diag]   batch 5/5: OK (50/50 done, llm_calls=0)
[diag] execute_prompt_batch done: llm_

In [7]:
report = optimizer.get_optimization_report()
print('Optimization generations summary:')
for entry in report['optimization_log']:
    print(f"  Generation {entry['generation']}: time {entry['time']:.2f}s, best_score {entry['best_score']:.3f}")

local_stats = report['component_statistics']['local_optimizer']
print('Local optimizer summary:')
print(f"  Total iterations recorded: {local_stats.get('total_iterations', 0)}")
avg_it = local_stats.get('avg_iteration_time')
if avg_it is not None:
    print(f"  Avg iteration time: {avg_it:.2f}s")
else:
    print('  Avg iteration time: N/A')
print(f"  Total LLM calls attributed to local iterations: {local_stats.get('total_llm_calls_by_local', 0)}")
print('Per-iteration breakdown:')
for s in local_stats.get('iteration_stats', []):
    print(f"  Iter {s['iteration']}: time {s['time']:.2f}s, llm_calls {s['llm_calls']}")

Optimization generations summary:
  Generation 1: time 1568.60s, best_score 0.760
  Generation 2: time 1188.75s, best_score 0.880
Local optimizer summary:
  Total iterations recorded: 4
  Avg iteration time: 625.31s
  Total LLM calls attributed to local iterations: 314
Per-iteration breakdown:
  Iter 1: time 402.05s, llm_calls 51
  Iter 2: time 1166.55s, llm_calls 178
  Iter 1: time 353.68s, llm_calls 29
  Iter 2: time 578.98s, llm_calls 56


In [8]:
print("BEST PROMPT FOUND:")
print("=" * 80)
print(best_node.prompt_text)
print("=" * 80)
print(f"Score: {best_node.metrics.composite_score():.3f}")
print(f"Generation: {best_node.generation}")
print(f"Source: {best_node.source.value}")

BEST PROMPT FOUND:
"Decompose the following math word problem into distinct steps. Identify the key numbers and operations needed to solve it. Perform the calculations accurately and clearly, and display the final numerical answer on the last line without any additional text, units, or symbols."
Score: 0.880
Generation: 3
Source: local


In [9]:
print("BASELINE vs OPTIMIZED — Test Set")
print("=" * 80)

comparison = optimizer.compare_with_baseline(initial_prompt, test_examples)

print()
print("Summary:")
print(f"  {'Metric':<25s}  {'Baseline':>10s}  {'Optimized':>10s}  {'Delta':>10s}")
print(f"  {'-'*25}  {'-'*10}  {'-'*10}  {'-'*10}")
for name in ["numeric_exact_match", "numeric_token_f1"]:
    b = comparison["baseline"].get(name, 0.0)
    o = comparison["optimized"].get(name, 0.0)
    d = comparison["improvements"].get(name, 0.0)
    print(f"  {name:<25s}  {b:10.3f}  {o:10.3f}  {d:+10.3f}")
b_c = comparison["baseline"]["composite_score"]
o_c = comparison["optimized"]["composite_score"]
print(f"  {'COMPOSITE':<25s}  {b_c:10.3f}  {o_c:10.3f}  {o_c - b_c:+10.3f}")

BASELINE vs OPTIMIZED — Test Set

Comparing with baseline...
[diag] evaluate_prompt: execute=True sample=True stage=3 incoming=50 will_use=50 weights={numeric_exact_match:0.8, numeric_token_f1:0.2} llm_calls_at_start=362
[diag] execute_prompt_batch: total=50 batch_size=10 n_batches=5 llm_calls_before=362
[diag]   batch 1/5: OK (10/50 done, llm_calls=363)
[diag]   batch 2/5: OK (20/50 done, llm_calls=364)
[diag]   batch 3/5: OK (30/50 done, llm_calls=365)
[diag]   batch 4/5: OK (40/50 done, llm_calls=366)
[diag]   batch 5/5: OK (50/50 done, llm_calls=367)
[diag] execute_prompt_batch done: llm_calls_delta=5 total_calls=367
[diag] evaluate_prompt execution done: llm_calls_for_execution=5
[diag]   example[0] expected='109' actual='82'
[diag]   example[1] expected='89' actual='66'
[diag]   example[2] expected='13' actual='10'
[diag]   ... and 47 more examples
[diag]   metric 'numeric_exact_match': 0.2600 (stage=1 weight=0.8 time=0.00s llm_calls=367)
[diag]   metric 'numeric_token_f1': 0.260

In [10]:
print(optimizer.visualize_optimization_trajectory())


OPTIMIZATION TRAJECTORY

Generation | Best Score | Overall Best | Improvement
------------------------------------------------------------
   1       | 0.760      | 0.760       | +0.540 ██████████████████████████████████████
   2       | 0.880      | 0.880       | +0.120 ████████████████████████████████████████████




In [11]:
report = optimizer.get_optimization_report()

print("OPTIMIZATION REPORT:")
print("Overall Statistics:")
print(f"   Total time: {report['optimization_info']['total_time_seconds']:.2f}s")
print(f"   Generations: {report['optimization_info']['generations']}")
print(f"   Total nodes explored: {report['component_statistics']['history']['total_nodes']}")

print("Component Statistics:")
print(f"   Local optimizer iterations: {report['component_statistics']['local_optimizer']['total_iterations']}")
print(f"   Local improvements: {report['component_statistics']['local_optimizer']['improvements_count']}")
print(f"   Global optimizer steps: {report['component_statistics']['global_optimizer']['total_global_steps']}")
print(f"   Successful global changes: {report['component_statistics']['global_optimizer']['successful_global_changes']}")

print("Best Global Strategies:")
for i, strategy in enumerate(report['best_global_strategies'][:3], 1):
    print(f"   {i}. Score {strategy['score']:.3f}")
    print(f"      {strategy['strategy']['description'][:70]}...")

OPTIMIZATION REPORT:
Overall Statistics:
   Total time: 2757.36s
   Generations: 2
   Total nodes explored: 19
Component Statistics:
   Local optimizer iterations: 4
   Local improvements: 3
   Global optimizer steps: 1
   Successful global changes: 0
Best Global Strategies:
   1. Score 0.400
      meta-optimizer-single...
   2. Score 0.240
      meta-optimizer-single...
   3. Score 0.240
      meta-optimizer-single...


In [12]:
lineage = optimizer.history.get_lineage(best_node.id)

print("EVOLUTION OF BEST PROMPT:")
print("=" * 80)

for i, node in enumerate(lineage):
    print(f"Step {i}: Generation {node.generation}, Source: {node.source.value}")
    if node.is_evaluated:
        print(f"  Score: {node.metrics.composite_score():.3f}")

    if node.operations:
        print(f"  Operations:")
        for op in node.operations:
            print(f"    - {op.description}...")

    if i < len(lineage) - 1:
        print("  ↓")

EVOLUTION OF BEST PROMPT:
Step 0: Generation 0, Source: initial
  Score: 0.220
  ↓
Step 1: Generation 1, Source: local
  Score: 0.700
  Operations:
    - MC synonym/paraphrase...
  ↓
Step 2: Generation 2, Source: local
  Score: 0.760
  Operations:
    - MC synonym/paraphrase...
  ↓
Step 3: Generation 3, Source: local
  Score: 0.880
  Operations:
    - MC synonym/paraphrase...


In [13]:
print("FINAL SUMMARY")
print("=" * 80)

history_stats = optimizer.history.get_statistics()
local_stats = optimizer.local_optimizer.get_statistics()
global_stats = optimizer.global_optimizer.get_statistics()

print("Overall Statistics:")
print(f"  Total nodes explored: {history_stats['total_nodes']}")
print(f"  Evaluations performed: {history_stats['evaluated_nodes']}")
print(f"  Generations completed: {history_stats['max_generation']}")
print(f"  Best score achieved: {history_stats['best_score']:.3f}")
print(f"  Average score: {history_stats['avg_score']:.3f}")

print("Local Optimization:")
print(f"  Total iterations: {local_stats['total_iterations']}")
print(f"  Improvements found: {local_stats['improvements_count']}")
print(f"  Success rate: {local_stats['improvement_rate']:.1%}")

print("Global Optimization:")
print(f"  Total global steps: {global_stats['total_global_steps']}")
print(f"  Candidates generated: {global_stats['total_candidates_generated']}")
print(f"  Successful changes: {global_stats['successful_global_changes']}")
print(f"  Success rate: {global_stats['success_rate']:.1%}")

print("\nOptimization complete!")
print(f"   Results saved to: ./optimization_results_gsm8k/")

FINAL SUMMARY
Overall Statistics:
  Total nodes explored: 19
  Evaluations performed: 19
  Generations completed: 3
  Best score achieved: 0.880
  Average score: 0.416
Local Optimization:
  Total iterations: 4
  Improvements found: 3
  Success rate: 75.0%
Global Optimization:
  Total global steps: 1
  Candidates generated: 8
  Successful changes: 0
  Success rate: 0.0%

Optimization complete!
   Results saved to: ./optimization_results_gsm8k/
